# Chapter 3 — Embeddings

Now we have numbers for every word. But a number like 9246 is
just a label. The model cannot learn from labels. It needs
meaning. It needs to know that cat and dog are similar and that
cat and car are different.

An embedding is a vector of floating point numbers that captures
the meaning of a word. Think of it like giving each word a
coordinate in a big space. Words with similar meanings end up
near each other. Words with different meanings end up far apart.

These embeddings start as random numbers. During training the
model learns to move related words closer together. After enough
training cat and dog will be neighbors while car will be on
the other side of the space. This notebook shows you how that
lookup table works and how the magic of learning makes it
meaningful.

## Imports

In [1]:
import torch
import torch.nn as nn
import math

## The Embedding Layer

In [2]:
class Embedding(nn.Module):
    def __init__(self, vocab_size, d_model):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, d_model)
        self.d_model = d_model

    def forward(self, x):
        return self.embed(x) * math.sqrt(self.d_model)

## Create an embedding table and test it

In [3]:
vocab_size = 1000
d_model = 768

embedding = Embedding(vocab_size, d_model)
print(f"Embedding table shape: [{vocab_size}, {d_model}]")
print(f"Total parameters: {sum(p.numel() for p in embedding.parameters()):,}")
print()

token_ids = torch.tensor([[12, 45, 678, 2, 890]])
print(f"Input token IDs: {token_ids}")
print(f"Input shape: {token_ids.shape}")
print()

output = embedding(token_ids)
print(f"Output shape: {output.shape}")
print(f"Each token is now a {d_model}-dimensional vector")
print(f"First token vector preview: {output[0, 0, :5].tolist()} ...")

Embedding table shape: [1000, 768]
Total parameters: 768,000

Input token IDs: tensor([[ 12,  45, 678,   2, 890]])
Input shape: torch.Size([1, 5])

Output shape: torch.Size([1, 5, 768])
Each token is now a 768-dimensional vector
First token vector preview: [18.91689682006836, -2.2418112754821777, 41.377845764160156, -2.377814531326294, 29.869359970092773] ...


## The scaling trick

We multiply embeddings by sqrt(d_model). This keeps the values
at a good scale before we add positional encoding later.
Without this scaling the position information would get lost.

In [4]:
embed_raw = nn.Embedding(vocab_size, d_model)
x = embed_raw(token_ids)

print(f"Without scaling: mean={x.mean():.4f}, std={x.std():.4f}")
print(f"With scaling: mean={(x * math.sqrt(d_model)).mean():.4f}, std={(x * math.sqrt(d_model)).std():.4f}")

Without scaling: mean=0.0087, std=0.9968
With scaling: mean=0.2402, std=27.6252


## Looking up embeddings for our tokenizer

Now we connect this to the real tokenizer from Chapter 2.

In [5]:
from dataclasses import dataclass
import tiktoken

@dataclass
class TokenizerConfig:
    name: str = "gpt2"
    vocab_size: int = 50257

class SimpleTokenizer:
    def __init__(self, config=None):
        self.config = config or TokenizerConfig()
        self.enc = tiktoken.get_encoding(self.config.name)
        self.eos_token = "<|endoftext|>"
        self.eos_token_id = self.enc.encode(
            self.eos_token, allowed_special={self.eos_token}
        )[0]
    def encode(self, text):
        return self.enc.encode(text, allowed_special={self.eos_token})
    def decode(self, ids):
        return self.enc.decode(ids)

tokenizer = SimpleTokenizer()
text = "The cat sat on the mat"
token_ids = tokenizer.encode(text)
print(f"Text: '{text}'")
print(f"Token IDs: {token_ids}")
print()

real_embedding = Embedding(tokenizer.vocab_size, d_model=768)
input_tensor = torch.tensor([token_ids])
vectors = real_embedding(input_tensor)
print(f"Output shape: {vectors.shape}")
print(f"Each of the {len(token_ids)} tokens is now a 768-dim vector")
print(f"The word 'cat' (token {token_ids[1]}) has vector: {vectors[0, 1, :3].tolist()} ...")

Text: 'The cat sat on the mat'
Token IDs: [464, 3797, 3332, 319, 262, 2603]



AttributeError: 'SimpleTokenizer' object has no attribute 'vocab_size'